# Complete JDBC Management Guide
Comprehensive JDBC operations matching standard CDAC style using `Scanner`, `PreparedStatement`, `Statement`, `CallableStatement`, `Transactions`, and `Batch Processing`.

### Database Configuration:
- **Database**: `pgcpac`
- **Table**: `student (id INT, name VARCHAR(50), mark INT)`
- **Driver**: `com.mysql.cj.jdbc.Driver`
- **URL**: `jdbc:mysql://localhost:3306/pgcpac`
- **User**: `root` | **Password**: `cdacacts`


---
## 0. JDBC Architecture, Drivers & Core Interfaces (Topics 124, 125 & 126)

### A. JDBC Architecture (Topic 124)
**Java Database Connectivity (JDBC)** is a standard Java API that enables Java programs to connect to relational database management systems (RDBMS):
```
  +-------------------------------------------------------------+
  |                     JAVA APPLICATION                        |
  +-------------------------------------------------------------+
                                 |
                                 v
  +-------------------------------------------------------------+
  |             JDBC API (java.sql & javax.sql)                 |
  |  [Connection]  [Statement]  [PreparedStatement] [ResultSet] |
  +-------------------------------------------------------------+
                                 |
                                 v
  +-------------------------------------------------------------+
  |              JDBC DRIVER MANAGER (DriverManager)            |
  +-------------------------------------------------------------+
                                 |
        +------------------------+-----------------------+
        |                        |                       |
        v                        v                       v
  [MySQL Driver]          [Postgres Driver]       [Oracle Driver] (JDBC Drivers)
        |                        |                       |
        v                        v                       v
  [MySQL Database]        [Postgres RDBMS]        [Oracle RDBMS]
```

### B. The 4 Types of JDBC Drivers (Topic 125)
1. **Type 1: JDBC-ODBC Bridge Driver**:
   - Translates JDBC calls to ODBC calls, which then communicate with the database.
   - *Drawback*: Dependent on native ODBC binary on client machine; poor performance; removed since Java 8.
2. **Type 2: Native-API Driver (Partly-Java Driver)**:
   - Translates JDBC calls directly into native client library vendor calls (e.g. OCI for Oracle).
   - *Drawback*: Requires vendor native client libraries installed on every client machine.
3. **Type 3: Network Protocol Driver (All-Java Middleware)**:
   - Sends JDBC calls over a generic network socket to an intermediate middleware server, which translates to database protocols.
   - *Advantage*: Highly flexible, no client-side database libraries required.
4. **Type 4: Thin Driver (Direct-to-Database Pure Java Driver)**:
   - Pure Java driver that connects directly to the database via vendor-specific socket protocols (e.g., `com.mysql.cj.jdbc.Driver`).
   - *Advantage*: Highest performance, 100% portable, zero client installation required. **Industry Standard**.

### C. Core JDBC Classes and Interfaces (Topic 126)
| Interface / Class | Type | Role & Responsibilities |
| :--- | :--- | :--- |
| **`DriverManager`** | Class | Manages a list of database drivers; establishes physical socket connections via `getConnection()` |
| **`Driver`** | Interface | Vendor-provided driver implementation loaded via `Class.forName()` |
| **`Connection`** | Interface | Represents an active session/connection with a specific database; creates statements and manages transactions |
| **`Statement`** | Interface | Executes static, non-parameterized SQL queries (susceptible to SQL Injection) |
| **`PreparedStatement`**| Interface | Pre-compiles parameterized SQL queries using `?` placeholders (fast, secure, reusable) |
| **`CallableStatement`** | Interface | Executes stored procedures and stored functions with `IN` and `OUT` parameters |
| **`ResultSet`** | Interface | Table cursor pointing to rows of data returned by query execution (`next()`, `getString()`, etc.) |
| **`DatabaseMetaData`** | Interface | Comprehensive metadata regarding the database software, capabilities, and catalogs |
| **`ResultSetMetaData`**| Interface | Metadata regarding the columns, names, and data types of a specific `ResultSet` |


---
## 1. Core CRUD Operations with `PreparedStatement`
Contains:
- `insertStudent()`: Read id, name, mark and insert into table.
- `selectAll()`: Fetch and display all student records.
- `selectById()`: Search and display a single record by id.
- `updateStudent()`: Update student mark by id.
- `deleteStudent()`: Delete a student record by id.


In [ ]:
package com.cdac;

import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.PreparedStatement;
import java.sql.ResultSet;
import java.sql.SQLException;
import java.util.Scanner;

public class JdbcManagement {

    // 1. INSERT STUDENT
    public void insertStudent() throws ClassNotFoundException, SQLException {
        Scanner s = new Scanner(System.in);
        System.out.println("enter the student id,name,mark");
        int n = s.nextInt();
        String na = s.next();
        int m = s.nextInt();

        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // create query
        String q = "insert into student values(?,?,?)";

        // create statement
        PreparedStatement pmt = con.prepareStatement(q);

        pmt.setInt(1, n);
        pmt.setString(2, na);
        pmt.setInt(3, m);

        int row = pmt.executeUpdate();
        System.out.println(row + " record inserted");

        con.close();
    }

    // 2. SELECT ALL STUDENTS
    public void selectAll() throws ClassNotFoundException, SQLException {
        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // create query
        String q = "select * from student";

        // create statement
        PreparedStatement pmt = con.prepareStatement(q);

        ResultSet rs = pmt.executeQuery();

        // iterate the result
        System.out.println("--- All Student Records ---");
        while (rs.next()) {
            int id = rs.getInt(1);
            String name = rs.getString(2);
            int mark = rs.getInt(3);
            System.out.println(id + " " + name + " " + mark);
        }

        con.close();
    }

    // 3. SELECT BY ID
    public void selectById() throws ClassNotFoundException, SQLException {
        Scanner s = new Scanner(System.in);
        System.out.println("enter the student id");
        int n = s.nextInt();

        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // create query
        String q = "select * from student where id = ?";

        // create statement
        PreparedStatement pmt = con.prepareStatement(q);

        pmt.setInt(1, n);

        ResultSet rs = pmt.executeQuery();

        // iterate the result
        while (rs.next()) {
            int id = rs.getInt(1);
            String name = rs.getString(2);
            int mark = rs.getInt(3);
            System.out.println(id + " " + name + " " + mark);
        }

        con.close();
    }

    // 4. UPDATE STUDENT
    public void updateStudent() throws ClassNotFoundException, SQLException {
        Scanner s = new Scanner(System.in);
        System.out.println("enter the student id,mark");
        int n = s.nextInt();
        int m = s.nextInt();

        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // create query
        String q = "update student set mark = ? where id = ?";

        // create statement
        PreparedStatement pmt = con.prepareStatement(q);

        pmt.setInt(1, m);
        pmt.setInt(2, n);

        int row = pmt.executeUpdate();
        System.out.println(row + " record updated");

        con.close();
    }

    // 5. DELETE STUDENT
    public void deleteStudent() throws ClassNotFoundException, SQLException {
        Scanner s = new Scanner(System.in);
        System.out.println("enter the student id");
        int n = s.nextInt();

        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // create query
        String q = "delete from student where id = ?";

        // create statement
        PreparedStatement pmt = con.prepareStatement(q);

        pmt.setInt(1, n);

        int row = pmt.executeUpdate();
        System.out.println(row + " record deleted");

        con.close();
    }
}


---
## 2. Table Creation Using Simple `Statement` (DDL)
Using static `Statement` for DDL queries like `CREATE TABLE`.


In [ ]:
package com.cdac;

import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.Statement;
import java.sql.SQLException;

public class TableCreation {

    public void createTable() throws ClassNotFoundException, SQLException {
        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // create query
        String q = "create table if not exists student (id int primary key, name varchar(50), mark int)";

        // create statement
        Statement stmt = con.createStatement();

        stmt.executeUpdate(q);
        System.out.println("student table created successfully");

        con.close();
    }
}


---
## 3. Transaction Management (`commit` and `rollback`)
Disable auto-commit, execute multiple operations atomically, commit on success, and rollback on any error.


In [ ]:
package com.cdac;

import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.PreparedStatement;
import java.sql.SQLException;
import java.util.Scanner;

public class TransactionManagement {

    public void transferMarks() throws ClassNotFoundException {
        Scanner s = new Scanner(System.in);
        System.out.println("enter source student id, target student id, marks to transfer");
        int fromId = s.nextInt();
        int toId = s.nextInt();
        int marks = s.nextInt();

        Connection con = null;
        try {
            // loading driver
            Class.forName("com.mysql.cj.jdbc.Driver");

            // get connection
            con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

            // 1. disable auto commit
            con.setAutoCommit(false);

            // 2. deduct mark from first student
            String q1 = "update student set mark = mark - ? where id = ?";
            PreparedStatement pmt1 = con.prepareStatement(q1);
            pmt1.setInt(1, marks);
            pmt1.setInt(2, fromId);
            pmt1.executeUpdate();

            // 3. add mark to second student
            String q2 = "update student set mark = mark + ? where id = ?";
            PreparedStatement pmt2 = con.prepareStatement(q2);
            pmt2.setInt(1, marks);
            pmt2.setInt(2, toId);
            pmt2.executeUpdate();

            // 4. commit if both queries succeed
            con.commit();
            System.out.println("transaction committed successfully");

        } catch (SQLException e) {
            // 5. rollback if any query fails
            try {
                if (con != null) {
                    con.rollback();
                    System.out.println("transaction rolled back due to error");
                }
            } catch (SQLException ex) {
                ex.printStackTrace();
            }
        } finally {
            try {
                if (con != null) {
                    con.close();
                }
            } catch (SQLException e) {
                e.printStackTrace();
            }
        }
    }
}


---
## 4. Batch Processing (`addBatch` & `executeBatch`)
Inserts multiple student records at once in a single network round-trip using `pmt.addBatch()`.


In [ ]:
package com.cdac;

import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.PreparedStatement;
import java.sql.SQLException;
import java.util.Scanner;

public class BatchProcessing {

    public void insertBatch() throws ClassNotFoundException, SQLException {
        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // create query
        String q = "insert into student values(?,?,?)";
        PreparedStatement pmt = con.prepareStatement(q);

        Scanner s = new Scanner(System.in);
        System.out.println("how many students you want to add in batch?");
        int count = s.nextInt();

        for (int i = 1; i <= count; i++) {
            System.out.println("enter id, name, mark for student " + i);
            int id = s.nextInt();
            String name = s.next();
            int mark = s.nextInt();

            pmt.setInt(1, id);
            pmt.setString(2, name);
            pmt.setInt(3, mark);

            // add to batch buffer
            pmt.addBatch();
        }

        // execute batch
        int[] rows = pmt.executeBatch();
        System.out.println(rows.length + " records inserted in batch successfully");

        con.close();
    }
}


---
## 5. Stored Procedures (`CallableStatement`)
Calling stored procedures in MySQL using `{call procedure_name(?, ?)}` with IN and OUT parameters.


In [ ]:
package com.cdac;

import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.CallableStatement;
import java.sql.Types;
import java.sql.SQLException;
import java.util.Scanner;

public class StoredProcedureDemo {

    public void getStudentMark() throws ClassNotFoundException, SQLException {
        Scanner s = new Scanner(System.in);
        System.out.println("enter the student id");
        int n = s.nextInt();

        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        // call procedure: getMark(IN sid INT, OUT smark INT)
        String q = "{call getMark(?, ?)}";

        CallableStatement cmt = con.prepareCall(q);

        // set IN parameter
        cmt.setInt(1, n);

        // register OUT parameter
        cmt.registerOutParameter(2, Types.INTEGER);

        // execute procedure
        cmt.execute();

        int resultMark = cmt.getInt(2);
        System.out.println("Student ID: " + n + " | Mark from Procedure: " + resultMark);

        con.close();
    }
}


---
### 5.1 Invoking Stored Functions via `CallableStatement` (Topic 128)
- **Stored Procedure vs. Stored Function**:
  - Stored Procedure: Called with `{call proc_name(?, ?)}`; may or may not return values via `OUT` parameters.
  - Stored Function: Called with **`{? = call function_name(?)}`**; **must** register parameter index 1 as return value via `registerOutParameter(1, Types.TYPE)`.


In [ ]:
// Invoking a Stored Function in MySQL via CallableStatement
// Syntax: {? = call function_name(param1, param2)}
import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.CallableStatement;
import java.sql.Types;
import java.sql.SQLException;

class StoredFunctionDemo {
    public static void callStudentMarksFunction(int studentId) {
        String url = "jdbc:mysql://localhost:3306/pgcpac";
        String user = "root";
        String pass = "cdacacts";
        String functionSql = "{? = call getStudentGrade(?)}";

        try (Connection con = DriverManager.getConnection(url, user, pass);
             CallableStatement cs = con.prepareCall(functionSql)) {

            // Parameter 1 is the RETURN value of the function
            cs.registerOutParameter(1, Types.VARCHAR);

            // Parameter 2 is the IN argument
            cs.setInt(2, studentId);

            // Execute function
            cs.execute();

            // Extract returned result
            String gradeResult = cs.getString(1);
            System.out.println("Function executed. Student " + studentId + " Grade: " + gradeResult);
        } catch (SQLException e) {
            System.out.println("[Stored Function Simulation]: Database function executed or offline: " + e.getMessage());
        }
    }
}

StoredFunctionDemo.callStudentMarksFunction(1);


---
## 6. Metadata (`ResultSetMetaData`)
Dynamically inspect table structure: column count, column names, and column types.


In [ ]:
package com.cdac;

import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.PreparedStatement;
import java.sql.ResultSet;
import java.sql.ResultSetMetaData;
import java.sql.SQLException;

public class MetaDataDemo {

    public void displayMetaData() throws ClassNotFoundException, SQLException {
        // loading driver
        Class.forName("com.mysql.cj.jdbc.Driver");

        // get connection
        Connection con = DriverManager.getConnection("jdbc:mysql://localhost:3306/pgcpac", "root", "cdacacts");

        String q = "select * from student";
        PreparedStatement pmt = con.prepareStatement(q);
        ResultSet rs = pmt.executeQuery();

        // get result set metadata
        ResultSetMetaData rsmd = rs.getMetaData();
        int count = rsmd.getColumnCount();
        System.out.println("Total Columns: " + count);

        for (int i = 1; i <= count; i++) {
            System.out.println("Column " + i + ": " + rsmd.getColumnName(i) + " (" + rsmd.getColumnTypeName(i) + ")");
        }

        con.close();
    }
}


---
### 6.1 Database MetaData (`DatabaseMetaData`) (Topic 129)
While `ResultSetMetaData` provides details about columns in a query result, **`DatabaseMetaData`** provides comprehensive information about the database software, server version, supported SQL features, and tables:
- Obtained via **`connection.getMetaData()`**.
- Useful methods: `getDatabaseProductName()`, `getDatabaseProductVersion()`, `getDriverName()`, `getDriverVersion()`, `getUserName()`, `getURL()`.


In [ ]:
import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.DatabaseMetaData;
import java.sql.ResultSet;
import java.sql.SQLException;

class DatabaseMetaDataDemo {
    public static void displayDatabaseProperties() {
        String url = "jdbc:mysql://localhost:3306/pgcpac";
        String user = "root";
        String pass = "cdacacts";

        try (Connection con = DriverManager.getConnection(url, user, pass)) {
            // Extract DatabaseMetaData
            DatabaseMetaData dbmd = con.getMetaData();

            System.out.println("--- DATABASE METADATA INFO ---");
            System.out.println("Database Product Name    : " + dbmd.getDatabaseProductName());
            System.out.println("Database Product Version : " + dbmd.getDatabaseProductVersion());
            System.out.println("JDBC Driver Name         : " + dbmd.getDriverName());
            System.out.println("JDBC Driver Version      : " + dbmd.getDriverVersion());
            System.out.println("Database URL             : " + dbmd.getURL());
            System.out.println("Connected User           : " + dbmd.getUserName());
            System.out.println("Supports Batch Updates?  : " + dbmd.supportsBatchUpdates());
            System.out.println("Supports Stored Procs?   : " + dbmd.supportsStoredProcedures());
        } catch (SQLException e) {
            System.out.println("[DatabaseMetaData Demo]: Database properties queried: " + e.getMessage());
        }
    }
}

DatabaseMetaDataDemo.displayDatabaseProperties();


---
## 7. SQL Injection Prevention & DAO Design Pattern

### A. SQL Injection Vulnerability (`Statement`) vs Prevention (`PreparedStatement`)
- **SQL Injection**: An attacker inputs malicious SQL fragments (e.g. `' OR '1'='1`) into input fields.
- **Why `Statement` is Vulnerable**: It concatenates raw input directly into the SQL string:
  ```java
  // DANGEROUS! If userInput is "' OR '1'='1", all records are exposed!
  String sql = "SELECT * FROM users WHERE user = '" + userInput + "'";
  ```
- **Why `PreparedStatement` is Safe**: Uses parameterized placeholders (`?`). The database compiles the query structure first, and user inputs are strictly bound as pure literal values—preventing executable SQL injection.

### B. DAO (Data Access Object) Design Pattern
Separates business logic from database persistence logic:
```
  [UI / Controller] ---- calls ----> [StudentDAO Service] ---- executes SQL ----> [Database (pgcpac)]
```


In [ ]:
// Clean DAO Pattern Demonstration Structure
class StudentRecordModel {
    int id;
    String name;
    int mark;

    public StudentRecordModel(int id, String name, int mark) {
        this.id = id;
        this.name = name;
        this.mark = mark;
    }
}

interface StudentDAOInterface {
    void save(StudentRecordModel student);
    StudentRecordModel findById(int id);
}

System.out.println("DAO Design Pattern decouples Application Logic from SQL Persistence!");
